In [1]:
from Database.transform import *
from Database.embed_models import AsyncSapBERTEmbedder
from dotenv import dotenv_values
import sys

c:\Users\Natt\miniconda3\envs\qd_llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
env = dotenv_values(r"H:\NatT\snomed\.env")
neo4j_dict = {
        "uri": env.get('NEO4J_HOST'),
        "username": env.get('NEO4J_USER'),
        "password": env.get('NEO4J_PASSWORD'),
        "snomed_release": env.get('SNOMED_RELEASE_PATH'),
}

In [3]:
c_, s_, r_ = reformat_snomed(data_path="H:/NatT/snomed/DATA/SnomedCT_InternationalRF2_PRODUCTION_20250301T120000Z/")
embedding_model = AsyncSapBERTEmbedder()

In [ ]:
async def main():
      if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
      await awrite_concept_nodes(
          lf=c_, 
          neo4j_url=neo4j_dict['uri'], 
          neo4j_user=neo4j_dict['username'], 
          neo4j_password=neo4j_dict['password'],
          batch_size = 10_000,
          )
      await awrite_relationships_apoc(r_, neo4j_dict['uri'], neo4j_dict['username'], neo4j_dict['password'])
      await awrite_description_nodes(
          lf= s_,
          embedding_model=embedding_model,
          neo4j_url=neo4j_dict['uri'],
          neo4j_user=neo4j_dict['username'],
          neo4j_password=neo4j_dict['password'],
          batch_size = 10_000,
      )
      

In [6]:
delete_all_in_batches(uri=neo4j_dict['uri'], user=neo4j_dict['username'], password=neo4j_dict['password'])

{'nodes_deleted': 1236263, 'relationships_deleted': 853020, 'loops': 43}

In [7]:
await main()

Writing Relationships:   0%|          | 0/1291604 [00:00<?, ?it/s]

Error writing relationships: {neo4j_code: Neo.ClientError.Procedure.ProcedureNotFound} {message: There is no procedure with the name `apoc.create.relationship` registered for this database instance. Please ensure you've spelled the procedure name correctly and that the procedure is properly deployed.} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}


In [4]:
await awrite_relationships(r_, neo4j_dict['uri'], neo4j_dict['username'], neo4j_dict['password'])

Writing Relationships: 100%|██████████| 1291604/1291604 [86:16:37<00:00,  4.16it/s]   

Skipped 0 invalid relationship rows.
